In [1]:
%load_ext autoreload
%autoreload 2

import edgar

from equibot.config import settings
from equibot.parsers import transcript
from etl.db import connect


edgar.set_identity(settings.sec_api_username)

with connect() as conn:                      # closes the connection on exit
    print(conn.execute("SELECT version()").fetchone()["version"])

c:\Users\benak\repos\EquiBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PostgreSQL 18.6 (Debian 18.6-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [2]:
from etl import ingest


In [3]:
goog = edgar.Company("GOOG")
filings = goog.get_filings(form="10-Q")

In [4]:
# add company
ingest.upsert_company(goog)

8

In [7]:
ingest.upsert_filing(goog, filings[0])

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1346.23it/s]
C:\Users\benak\repos\laya\laya\router.py:999: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


  skipped 274 facts with no period end


6

In [9]:
filing = filings[0]
tenq = filing.obj()

In [18]:
filing.xbrl().statements.list_available()

,index,category,name,role_name,element_count
0,0,document,0000001 - Document - COVER PAGE,COVERPAGE,62
1,1,statement,9952151 - Statement - CONSOLIDATED BALANCE SHEETS,CONSOLIDATEDBALANCESHEETS,40
2,2,statement,9952152 - Statement - CONSOLIDATED BALANCE SHE...,CONSOLIDATEDBALANCESHEETSParenthetical,19
3,3,statement,9952153 - Statement - CONSOLIDATED STATEMENTS ...,CONSOLIDATEDSTATEMENTSOFINCOME,17
4,4,statement,9952154 - Statement - CONSOLIDATED STATEMENTS ...,CONSOLIDATEDSTATEMENTSOFCOMPREHENSIVEINCOME,14
...,...,...,...,...,...
84,84,disclosure,AwardTimingDisclosure,AwardTimingDisclosure,20
85,85,disclosure,ErrCompDisclosure,ErrCompDisclosure,20
86,86,disclosure,PvpDisclosure,PvpDisclosure,53
87,87,note,InsiderTradingArrangements,InsiderTradingArrangements,28


In [20]:
filing.xbrl().statements.get('COVER PAGE').to_dataframe()

,concept,label,standard_concept,2026-09-30 (YTD),level,abstract,dimension,is_breakdown,dimension_axis,dimension_member,dimension_member_label,dimension_label,balance,weight,preferred_sign,parent_concept,parent_abstract_concept
0,dei_CoverAbstract,Cover page.,None,NaN,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN
1,dei_DocumentType,Document Type,None,10-Q,3,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,1.0,None,dei_DocumentInformationLineItems
2,dei_DocumentQuarterlyReport,Document Quarterly Report,None,true,3,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,1.0,None,dei_DocumentInformationLineItems
3,dei_DocumentPeriodEndDate,Document Period End Date,None,2026-09-30,3,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,1.0,None,dei_DocumentInformationLineItems
4,dei_DocumentTransitionReport,Document Transition Report,None,false,3,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,1.0,None,dei_DocumentInformationLineItems
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,dei_AmendmentFlag,Amendment Flag,None,false,3,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,1.0,None,dei_DocumentInformationLineItems
108,dei_DocumentFiscalYearFocus,Document Fiscal Year Focus,None,2026,3,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,1.0,None,dei_DocumentInformationLineItems
109,dei_DocumentFiscalPeriodFocus,Document Fiscal Period Focus,None,Q3,3,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,1.0,None,dei_DocumentInformationLineItems
110,dei_EntityCentralIndexKey,Entity Central Index Key,None,0001652044,3,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,1.0,None,dei_DocumentInformationLineItems


In [24]:
filing.report_date

'2026-09-30'

In [25]:
filing.filing_date

datetime.date(2026, 10, 29)

In [116]:
from pydantic_ai import Agent

agent = Agent(model=model)
result = await agent.run("How much wood can a woodchuck chuck?")
print(result.output)

Ah, the classic tongue twister! The traditional answer is: "A woodchuck would chuck as much wood as a woodchuck could chuck if a woodchuck could chuck wood."

Of course, in reality, woodchucks (also known as groundhogs) don't actually chuck wood at all—they're burrowing rodents that dig tunnels and eat vegetation. But if you want a semi-scientific answer: wildlife expert Richard Thomas famously calculated in 1988 that, based on the volume of dirt a woodchuck moves when digging its burrow, a woodchuck could hypothetically chuck about 700 pounds of wood.

So there you have it—somewhere between "none" and "roughly 700 pounds," depending on how seriously you take the question!


In [120]:
res = {}
for s in tenq.sections:
    res[s] = await agent.run(
        f"The following is a section of a {tenq.form} financial report with item number {s}:\n"
        f"Please read the section and provide a title for it along with a brief summary.\n"
        f"Ex. 'Title: Management's Discussion and Analysis, Summary: This section provides an overview of the company's financial performance and key metrics.'\n"
        f"Section Text:\n{tenq.sections.get(s).text()}"
    )